# SVD of flow past a cylinder

We analyze a data matrix of fluid flow past a cylinder, where each column is a flow field reshaped into a vector. The goals are:

1. compute the SVD of the flow data and interpret the matrices $U$, $\Sigma$, and $V^T$,
2. reconstruct the flow for different truncation ranks and compare the fidelity,
3. compute energy-based truncation ranks,
4. verify the interpretation of the coordinates $w_k$ in the reduced space,
5. build a linear regression model for the reduced amplitudes,
6. use that model to forecast future reduced states and flow snapshots.


## Learning goals

By the end of this notebook, you should be able to:

- compute the economy SVD of a flow snapshot matrix,
- interpret left singular vectors as dominant spatial flow structures,
- interpret $\Sigma V^T$ as time-dependent amplitudes,
- choose truncation ranks based on cumulative energy,
- measure reconstruction error with the Frobenius norm,
- verify the representation $x_k \approx \widetilde U w_k$,
- fit a linear dynamical model in reduced coordinates.


## 1. Load the reduced cylinder-flow data

We use a dataset stored as `data/cylinder_vorticity.npz`.

This file contains:

- `X`: the vorticity snapshot matrix,
- `nx`: number of grid points in the horizontal direction,
- `ny`: number of grid points in the vertical direction.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_cylinder_data():
    data = np.load("data/cylinder_vorticity.npz")
    X = np.asarray(data["X"], dtype=np.float32)
    nx = int(np.array(data["nx"]).squeeze())
    ny = int(np.array(data["ny"]).squeeze())

    return X, nx, ny

def plot_snapshot(vec, nx, ny, title="Snapshot", cmap="seismic"):
    plt.figure(figsize=(5, 4))
    field = vec.reshape(ny, nx)
    vmax = np.max(np.abs(field))
    plt.imshow(field, cmap=cmap, origin="lower", aspect="auto", vmin=-vmax, vmax=vmax)
    plt.colorbar()
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
X, nx, ny = load_cylinder_data()

print("Snapshot matrix shape:", X.shape)
print("nx =", nx, "ny =", ny)


## 2. Visualize a few flow snapshots

We first inspect several columns of the data matrix to understand what one snapshot looks like and how the vorticity field evolves in time.

Since `nx` and `ny` are provided by the dataset, each snapshot can be reshaped into a two-dimensional field.


In [ ]:
indices = [0, X.shape[1] // 4, X.shape[1] // 2, 3 * X.shape[1] // 4]

for k in indices:
    plot_snapshot(X[:, k], nx=nx, ny=ny, title=f"Snapshot {k}")


## 3. Compute the SVD of the flow data

We compute the economy SVD
$$
X = U\Sigma V^T.
$$

Here:
- the columns of $U$ are dominant spatial flow structures,
- the singular values in $\Sigma$ measure their importance,
- the rows of $\Sigma V^T$ contain the time-dependent amplitudes.


In [ ]:
U, s, VT = np.linalg.svd(X, full_matrices=False)
Sigma = np.diag(s)

print(U.shape, s.shape, VT.shape)


## 4. Plot the singular value spectrum and leading modes

We now visualize:
- the singular value spectrum,
- the cumulative energy,
- the first few left singular vectors.

If `nx` and `ny` are known, the left singular vectors are reshaped into spatial flow fields.


In [ ]:
energy = s**2
cum_energy = np.cumsum(energy) / np.sum(energy)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].semilogy(s, "k")
axes[0].set_title("Singular value spectrum")
axes[0].set_xlabel("index")
axes[0].set_ylabel("singular value")

axes[1].plot(cum_energy, "k")
axes[1].set_title("Cumulative energy")
axes[1].set_xlabel("index")
axes[1].set_ylabel("fraction of total energy")

plt.tight_layout()
plt.show()


In [ ]:
r_plot = 6
fig, axes = plt.subplots(1, r_plot, figsize=(14, 3.5))

for j in range(r_plot):
    if nx is not None and ny is not None and nx * ny == U[:, j].size:
        field = U[:, j].reshape(ny, nx)
        axes[j].imshow(field, cmap="seismic", origin="lower", aspect="auto")
        axes[j].axis("off")
    else:
        axes[j].plot(U[:, j], color="k")
    axes[j].set_title(f"Mode {j+1}")

plt.tight_layout()
plt.show()


## 5. Reconstruct the flow with different truncation ranks

We reconstruct the snapshot matrix using truncated SVD:
$$
\widetilde X_r = U_r \Sigma_r V_r^T.
$$

We use ranks chosen to capture:
- 90% of the total energy,
- 99% of the total energy,
- 99.9% of the total energy.

Recall that the energy is based on the squared singular values.


In [ ]:
energy = s**2
cumulative_energy = np.cumsum(energy) / np.sum(energy)

r90 = int(np.searchsorted(cumulative_energy, 0.90) + 1)
r99 = int(np.searchsorted(cumulative_energy, 0.99) + 1)
r999 = int(np.searchsorted(cumulative_energy, 0.999) + 1)

print("r90  =", r90)
print("r99  =", r99)
print("r999 =", r999)


In [ ]:
def truncated_svd_reconstruction(U, s, VT, r):
    return U[:, :r] @ np.diag(s[:r]) @ VT[:r, :]

X90 = truncated_svd_reconstruction(U, s, VT, r90)
X99 = truncated_svd_reconstruction(U, s, VT, r99)
X999 = truncated_svd_reconstruction(U, s, VT, r999)


In [ ]:
err90 = np.linalg.norm(X - X90, ord="fro")**2
err99 = np.linalg.norm(X - X99, ord="fro")**2
err999 = np.linalg.norm(X - X999, ord="fro")**2

print("Squared Frobenius errors:")
print("90%   :", err90)
print("99%   :", err99)
print("99.9% :", err999)


In [ ]:
k = X.shape[1] // 2

plot_snapshot(X[:, k], nx=nx, ny=ny, title="Original snapshot")
plot_snapshot(X90[:, k], nx=nx, ny=ny, title=f"90% energy (r={r90})")
plot_snapshot(X99[:, k], nx=nx, ny=ny, title=f"99% energy (r={r99})")
plot_snapshot(X999[:, k], nx=nx, ny=ny, title=f"99.9% energy (r={r999})")

#To save memory
del X90, X999
import gc
gc.collect()

## 6. Fix $r=10$ and interpret the reduced coordinates

Let
$$
\widetilde X = \widetilde U \widetilde \Sigma \widetilde V^T,
$$
with $r=10$, and define
$$
W = \widetilde \Sigma \widetilde V^T.
$$

Each column $w_k \in \mathbb{R}^{10}$ contains the amplitudes of the first 10 eigenflows in snapshot $k$.
We verify that
$$
x_k \approx \widetilde U w_k.
$$


In [ ]:
r = 10

U_tilde = U[:, :r]
Sigma_tilde = np.diag(s[:r])
VT_tilde = VT[:r, :]
W = Sigma_tilde @ VT_tilde

print("W shape:", W.shape)


In [ ]:
k = X.shape[1] // 3
xk = X[:, k]
wk = W[:, k]
xk_recon = U_tilde @ wk

rel_err_k = np.linalg.norm(xk - xk_recon) / np.linalg.norm(xk)
print("Relative reconstruction error:", rel_err_k)

plot_snapshot(xk, nx=nx, ny=ny, title=f"Original snapshot {k}")
plot_snapshot(xk_recon, nx=nx, ny=ny, title=f"U_tilde @ w_{k}")


## 7. Fit a linear dynamical model in reduced coordinates

We now build the regression model
$$
w_{k+1} = A w_k.
$$

If we define
$$
W_1 = [w_1, w_2, \dots, w_{m-1}], \qquad
W_2 = [w_2, w_3, \dots, w_m],
$$
then the least-squares matrix is
$$
A = W_2 W_1^\dagger.
$$


In [ ]:
W1 = W[:, :-1]
W2 = W[:, 1:]
A_red = W2 @ np.linalg.pinv(W1)

print(A_red.shape)


In [ ]:
W2_pred = A_red @ W1
rel_err_reduced = np.linalg.norm(W2 - W2_pred, ord="fro") / np.linalg.norm(W2, ord="fro")

print("Relative reduced-state one-step error:", rel_err_reduced)


In [ ]:
n_forecast = min(20, W.shape[1] - 1)
w_forecast = np.zeros((r, n_forecast + 1))
w_forecast[:, 0] = W[:, 0]

for j in range(n_forecast):
    w_forecast[:, j + 1] = A_red @ w_forecast[:, j]

j = n_forecast
x_pred = U_tilde @ w_forecast[:, j]
x_true = X[:, j]

plot_snapshot(x_true, nx=nx, ny=ny, title=f"True snapshot {j}")
plot_snapshot(x_pred, nx=nx, ny=ny, title=f"Predicted snapshot {j}")


## 8. Animate the original and reconstructed flows

A movie often gives a better sense of the coherent structures in the wake than static snapshots.

In this section, we build:

- one comparison movie with **original** and **reconstructed** flow shown side by side,
- one separate movie for the **reconstruction error**.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_two_flow_comparison(X_left, X_right, nx, ny,
                                title_left="Original",
                                title_right="Reconstructed",
                                cmap="seismic", interval=50):
    """
    Return a JavaScript animation with two side-by-side subplots
    sharing the same color scale.
    """
    n_frames = min(X_left.shape[1], X_right.shape[1])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    vmax = max(
        np.max(np.abs(X_left)),
        np.max(np.abs(X_right)),
    )

    mats = [X_left, X_right]
    titles = [title_left, title_right]
    ims = []

    for ax, Xmat, ttl in zip(axes, mats, titles):
        field0 = Xmat[:, 0].reshape(ny, nx)
        im = ax.imshow(
            field0,
            cmap=cmap,
            origin="lower",
            aspect="auto",
            vmin=-vmax,
            vmax=vmax,
        )
        ax.set_title(f"{ttl}: frame 0")
        ax.axis("off")
        ims.append(im)

    fig.colorbar(ims[-1], ax=axes.ravel().tolist(), shrink=0.8)

    def update(frame):
        for ax, im, Xmat, ttl in zip(axes, ims, mats, titles):
            field = Xmat[:, frame].reshape(ny, nx)
            im.set_array(field)
            ax.set_title(f"{ttl}: frame {frame}")
        return ims

    anim = FuncAnimation(fig, update, frames=n_frames, interval=interval, blit=True)
    plt.close(fig)
    return anim

def animate_error_flow(X_err, nx, ny, title="Error", cmap="seismic", interval=50):
    """
    Return a JavaScript animation for the error field alone, using its own color scale.
    """
    n_frames = X_err.shape[1]

    fig, ax = plt.subplots(figsize=(5.5, 4))

    vmax = np.max(np.abs(X_err))
    field0 = X_err[:, 0].reshape(ny, nx)

    im = ax.imshow(
        field0,
        cmap=cmap,
        origin="lower",
        aspect="auto",
        vmin=-vmax,
        vmax=vmax,
    )
    ax.set_title(f"{title}: frame 0")
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.8)

    def update(frame):
        field = X_err[:, frame].reshape(ny, nx)
        im.set_array(field)
        ax.set_title(f"{title}: frame {frame}")
        return [im]

    anim = FuncAnimation(fig, update, frames=n_frames, interval=interval, blit=True)
    plt.close(fig)
    return anim


We compare the original movie with the 99% energy reconstruction, and then animate the reconstruction error separately.


In [ ]:
# Choose one reconstructed matrix
X_recon = X99
X_err = X - X_recon


In [ ]:
# Create movies
anim_compare = animate_two_flow_comparison(
    X, X_recon, nx, ny,
    title_left="Original flow",
    title_right="99% reconstruction",
)

anim_error = animate_error_flow(
    X_err, nx, ny,
    title="Reconstruction error",
)


In [ ]:
# Display original vs reconstruction
HTML(anim_compare.to_jshtml())


In [ ]:
# Display reconstruction error
HTML(anim_error.to_jshtml())


## 9. Animate the predicted flow from the reduced linear model

We now use the reduced linear regression model
$$
w_{k+1} = A w_k
$$
to generate predicted reduced states, map them back to the full flow space, and build:

- one comparison movie with **true** and **predicted** flow shown side by side,
- one separate movie for the **prediction error**.

Again, the separate error movie makes the smaller-scale error structures easier to visualize.


In [ ]:
# Choose how many frames to predict
n_pred = min(30, X.shape[1])

# Build predicted reduced states
W_pred = np.zeros((r, n_pred))
W_pred[:, 0] = W[:, 0]
for j2 in range(n_pred - 1):
    W_pred[:, j2 + 1] = A_red @ W_pred[:, j2]


In [ ]:
# Map back to the full flow space
X_pred = U_tilde @ W_pred

# Matching true snapshots and prediction error
X_true_pred = X[:, :n_pred]
X_pred_err = X_true_pred - X_pred

rel_pred_err = np.linalg.norm(X_true_pred - X_pred, ord="fro") / np.linalg.norm(X_true_pred, ord="fro")
print("Relative forecast error over displayed frames:", rel_pred_err)


In [ ]:
# Create movies
anim_pred_compare = animate_two_flow_comparison(
    X_true_pred, X_pred, nx, ny,
    title_left="True flow",
    title_right="Predicted flow",
)

anim_pred_error = animate_error_flow(
    X_pred_err, nx, ny,
    title="Prediction error",
)


In [ ]:
# Display true vs predicted
HTML(anim_pred_compare.to_jshtml())


In [ ]:
# Display prediction error
HTML(anim_pred_error.to_jshtml())


## Final discussion

1. **What do the leading singular vectors represent in this fluid-flow dataset?**  
   They represent dominant spatial flow structures, or eigenflow fields, that capture the most energetic patterns in the wake.

2. **How many modes are needed to capture 90%, 99%, and 99.9% of the total energy?**  
   This depends on the dataset, but the required ranks come directly from the cumulative sum of the squared singular values.

3. **Why does the approximation improve as the truncation rank increases?**  
   Because more singular components are retained, so the reconstruction includes more of the flow structure present in the original data.

4. **What is the meaning of the reduced coordinates $w_k$?**  
   They are the amplitudes of the retained eigenflow fields in the $k$th snapshot.

5. **Why is it reasonable to model the reduced coordinates with a linear system?**  
   After dimensionality reduction, the dominant dynamics are often smoother and lower-dimensional, so a linear regression model can capture part of their evolution.

6. **Where do you expect the regression model to perform well, and where might it fail?**  
   It should work best for short-term prediction and for dominant coherent structures. It may fail over longer horizons or when nonlinear effects become important.
